In [1]:
Output = '/Users/alexis/Library/CloudStorage/OneDrive-UniversityofNorthCarolinaatChapelHill/CEMALB_DataAnalysisPM/Projects/P1005. Miscellaneous Analyses/P1005.8. Wildfire As Review/Output'
cur_date = '062226'

library(readxl)
library(tidyverse)
library(stringr)
library(reshape2)
library(ggnewscale)

# reading in files
heatmap_data = read_csv("Input/Heatmap.csv")
water_info_df = data.frame(read_excel("Input/Extraction Table.xlsx"))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths


Rows: 22 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (3): Title, Citation, Location
dbl (10): Study_ID, Impacted_by_Soil, Impacted_by_Vegetation, Impacted_by_As...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet th

In [2]:
# head(heatmap_data)
# combining data
full_df = inner_join(heatmap_data, water_info_df[,c(1,2,9,11)])
head(full_df)

Joining with `by = join_by(Title, Citation)`


Study_ID,Title,Citation,Location,Impacted_by_Soil,Impacted_by_Vegetation,Impacted_by_Ash,Impacted_by_Smoke,Impacted_by_Built_Environment,Impacted_by_Land_Use,Pre_Wildfire_Arsenic_Concentration,Impacts_Surface_Water,Impacts_Groundwater,Primary_Mobilization_Pathway,Impacted_Water_Resource
<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,"(Murphy et al., 2020)",Colorado,1,0,1,0,0,1,1,1,0,Runoff,Surface
2,Feeding inhibition following in-situ and laboratory exposure as an indicator of ecotoxic impacts of wildfires in affected waterbodies,"(Ré et al., 2020)",Portugal,1,0,1,0,0,0,0,1,0,Runoff,Surface
3,"Paleolimnological Assessment of Wildfire-Derived Atmospheric Deposition of Trace Metal(loid)s and Major Ions to Subarctic Lakes (Northwest Territories, Canada)","(Pelletier et al., 2020)",Canada,0,0,1,0,0,0,1,1,0,Deposition,Surface
4,Assessment of superficial water quality of small catchment basins affected by Portuguese rural fires of 2017,"(Sequeira et al., 2020)",Portugal,1,0,1,0,0,0,1,1,0,Runoff,Surface
5,"Trace elements in stormflow, ash, and burned soil following the 2009 Station Fire in southern California","(Burton et al., 2016)",California,1,0,1,0,0,0,1,1,0,Runoff,Surface
6,The Rural Fires of 2017 and Their Influences on Water Quality: An Assessment of Causes and Effects,"(Sequeira et al., 2023)",Portugal,1,1,0,0,0,0,0,1,0,Runoff,Surface


Words about what the heatmap is showing.

In [3]:
heatmap_df = full_df %>%
    unite("location_citation", c("Location", "Citation"), sep = " ") %>%
    pivot_longer(cols = 4:12, names_to = "Variable", values_to = "Reported") %>%
    # cleaning up variable names
    mutate(Variable = str_replace_all(Variable, "_", " "),
          Variable = str_replace_all(Variable, "Arsenic", "As"),
          location_citation = str_replace_all(location_citation, ",", "")) %>%
    mutate(Reported = ifelse(Reported == 1, "Yes", "No"))

head(heatmap_df)

Study_ID,Title,location_citation,Primary_Mobilization_Pathway,Impacted_Water_Resource,Variable,Reported
<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Soil,Yes
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Vegetation,No
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Ash,Yes
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Smoke,No
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Built Environment,No
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Land Use,Yes


In [4]:
# ordering the df based on avg media/ fuel types and study averages
ordered_var_df = heatmap_df %>%
    group_by(Variable) %>%
    mutate(Count = ifelse(Reported == "Yes", 1, 0)) %>%
    summarize(Avg_Count = mean(Count)) %>%
    arrange(Avg_Count)

heatmap_df$Primary_Mobilization_Pathway = factor(heatmap_df$Primary_Mobilization_Pathway, levels = c('Likely deposition', 'Deposition',
                                                                                                    'Likely leaching', 'Leaching',
                                                                                                    'Likely runoff', 'Runoff'))
ordered_study_df = heatmap_df %>%
    group_by(Impacted_Water_Resource, Primary_Mobilization_Pathway) %>%
    arrange(Primary_Mobilization_Pathway) %>%
    mutate(Count = ifelse(Reported == "Yes", 1, 0)) 

# putting into factors for ordering
#heatmap_df$Variable = factor(heatmap_df$Variable, levels = ordered_var_df$Variable)
heatmap_df$location_citation = factor(heatmap_df$location_citation, levels = unique(ordered_study_df$location_citation))
heatmap_df$Reported = factor(heatmap_df$Reported, levels = c('Yes', 'No'))

head(heatmap_df)

Study_ID,Title,location_citation,Primary_Mobilization_Pathway,Impacted_Water_Resource,Variable,Reported
<dbl>,<chr>,<fct>,<fct>,<chr>,<chr>,<fct>
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Soil,Yes
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Vegetation,No
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Ash,Yes
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Smoke,No
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Built Environment,No
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Land Use,Yes


In [5]:
# creatinga annotation dfs
annotation_df <- unique(heatmap_df[, c("location_citation", "Impacted_Water_Resource")])
annotation_df2 <- unique(heatmap_df[, c("location_citation", "Primary_Mobilization_Pathway")])

# creating a df that leaves space b/w the annotation rows and the rest of the heatmap
spacer_df <- annotation_df
spacer_df$Variable <- "Spacer"
spacer_df$Primary_Mobilization_Pathway <- NA

annotation_df$Variable <- "Water"
annotation_df2$Variable <- "Pathway"

all_levels <- c("Water", "Pathway", "Spacer", unique(ordered_var_df$Variable))

heatmap_df$Variable <- factor(heatmap_df$Variable, levels = all_levels)
annotation_df$Variable <- factor(annotation_df$Variable, levels = all_levels)
annotation_df2$Variable <- factor(annotation_df2$Variable, levels = all_levels)
spacer_df$Variable <- factor(spacer_df$Variable, levels = all_levels)

head(heatmap_df)

Study_ID,Title,location_citation,Primary_Mobilization_Pathway,Impacted_Water_Resource,Variable,Reported
<dbl>,<chr>,<fct>,<fct>,<chr>,<fct>,<fct>
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Soil,Yes
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Vegetation,No
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Ash,Yes
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Smoke,No
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Built Environment,No
1,Wildfire-driven changes in hydrology mobilize arsenic and metals from legacy mine waste,Colorado (Murphy et al. 2020),Runoff,Surface,Impacted by Land Use,Yes


In [6]:
options(repr.plot.width=16, repr.plot.height=8) #changing size

Figure3 = ggplot() +

  # Annotation Row Layer
  geom_tile(data = annotation_df, aes(x = location_citation, y = Variable, fill = Impacted_Water_Resource),
            color = "white", linewidth = 0.5) + 
  scale_fill_manual(values = c("Ground" = "#d68b00", "Surface" = "#2B1055"),
                   name = "Impacted Water\n      Resource") + 
  # making the levels in the legend into rows
  guides(fill = guide_legend(nrow = 2)) +

  # Clear the fill scale to start a brand new color scale 
  new_scale_fill() +

  geom_tile(data = annotation_df2, aes(x = location_citation, y = Variable, fill = Primary_Mobilization_Pathway),
            color = "white", linewidth = 0.5) + 
    scale_fill_manual(values = c("Likely deposition" = "#90D5FF", "Deposition" = "#4682B4",
                              "Likely leaching" = "#B8E5B8", "Leaching" = "#636B2F",
                               "Likely runoff" = "#F7D9BC", "Runoff" = "#AB9682"),
                   name = "Primary Moblization\n         Pathway") + 

  # Clear the fill scale to start a brand new color scale 
  new_scale_fill() +

  # plotting the space
  geom_tile(data = spacer_df, aes(x = location_citation, y = Variable), fill = NA, color = NA) +

  geom_tile(data = heatmap_df, aes(x = location_citation, y = Variable, fill = Reported),
            color = "white", linewidth = 0.5) +
    scale_fill_manual(values = c("Yes" = "#f3c30a", "No" = "gray60"),
                   name = "Reported Pre-Wildfire\n   As Concentration") + 
  guides(fill = guide_legend(nrow = 2)) +

  theme_minimal() +

  theme(axis.title = element_text(face = "bold", size = rel(1.9)),
    axis.text.x = element_text(size = 13, angle = 45, hjust = 1),
    axis.text.y = element_text(size = 13), 
    legend.title = element_text(face = "bold", size = 15),
    legend.text = element_text(size = 13), #changes legend text
    legend.position = "bottom",
    legend.background = element_rect(colour = 'black', fill = 'white', linetype = 'solid')) +

  # removing labels from y axis
  scale_y_discrete(labels = function(x) ifelse(x %in% c("Spacer","Pathway", "Water"), "", x)) + 

  labs(y = "Impacted Media and\nFuel Type", x = "Study Location (Reference)") +

  # collecting all the legends together
  plot_layout(guides = "collect") &
      theme(legend.position = "bottom",
      legend.box = "horizontal") &
      guides(fill = guide_legend(nrow = 2))

Figure3

ERROR: Error in plot_layout(guides = "collect"): could not find function "plot_layout"


In [ ]:
# # exporting figure
# ggsave(Figure3, 
#        filename = 'Figure3.pdf',
#        path = Output,
#        width = 15, height = 7)